## Imports

In [51]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import matplotlib as mpl
from matplotlib import pyplot as pyplot
import json
import os
import copy
from numpy import random as rng

from misc import move

In [52]:
rng = np.random.default_rng()

## Load the data to visualize it

In [53]:
all_data_matrix = pd.read_csv('../Data/raw_data/all_subjects.csv')

In [54]:
all_data_matrix = all_data_matrix.drop('t',axis=1)
all_data_matrix = all_data_matrix[all_data_matrix['event'] != 'BONUS_FAIL']
all_data_matrix = all_data_matrix[all_data_matrix['event'] != 'BONUS_SUCCESS']
all_data_matrix = all_data_matrix[all_data_matrix['subject'] != 'debugmCrtV:debuglkTAQ']

In [55]:
subject_1 = all_data_matrix[all_data_matrix['subject']=='A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB']

In [56]:
s1_prb10206_7 = subject_1[subject_1['instance']=='prb10206_7']

In [57]:
s1_prb10206_7

,subject,event,move,instance,piece,target
5180,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb10206_7,-1,-1
5181,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,0,prb10206_7,4,2
5182,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,0,prb10206_7,4,3
5183,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,1,prb10206_7,3,8
5184,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,1,prb10206_7,3,2
...,...,...,...,...,...,...
17231,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,10,prb10206_7,8,16
17232,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,11,prb10206_7,8,16
17233,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,11,prb10206_7,8,16
17234,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,11,prb10206_7,8,16


In [91]:
def parse_data_matrix(df):
    df = copy.deepcopy(df)
    df['sq_change'] = df.shift(-1)['target'] - df['target']
    df['dist'] = (df['sq_change'])*((df['sq_change'] % 6) != 0) + (df['sq_change'] // 6)*((df['sq_change'] % 6) == 0)
    df['piece'] = df['piece'].where(df['piece'] != 8, -1)
    df['piece'] = df['piece'].where(df['piece'] == -1, df['piece'] + 1)
    df.loc[df['event'].isin(pd.Series(['win','start','restart','surrender'])), ['piece','target','sq_change','dist']] = None
    df['piece'] = df['piece'].astype('Int64')
    df['p_as_str'] = df['piece'].astype('str')
    df = df[df['event'] != 'drag_end']
    df = df[~((df['event'] == 'win') & (df['event'].shift() == 'win'))]
    df = df[df['dist'] != 0]
    df['dist'] = df['dist'].astype('Int64')
    df.loc[df['event'] == 'drag_start','event'] = 'move'
    df = df.drop(columns=['target','sq_change','piece'])
    df = df.fillna(0)
    df['solve_instance'] = ((df['event'] == 'start') & (df['event'].shift().isin(pd.Series(['win','surrender'])))).cumsum()
    tot_solve_insts = pd.unique(df['solve_instance'])
    frames = []
    for instance in tot_solve_insts:
        dup = False
        view = df[df['solve_instance']==instance].reset_index().drop(columns = ['index','solve_instance'])
        for f in frames:
            if view.equals(f):
                dup = True
                break
        if not dup:
            frames.append(view)

    no_dup_df = pd.concat(frames)
        
    return no_dup_df

In [92]:
s1_prb10206_7_processed = parse_data_matrix(s1_prb10206_7)
s1_prb10206_7_processed.to_csv('../Data/my_processed_data/s1_prb10206_7_processed.csv')

In [95]:
subject_1_p = parse_data_matrix(subject_1)
first_g_first_instc = subject_1_p[subject_1_p['instance']=='prb55384_14']
first_g_first_instc

,subject,event,move,instance,dist,p_as_str
0,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb55384_14,0,0
1,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb55384_14,1,3
2,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,1,prb55384_14,-2,3
3,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,2,prb55384_14,2,3
4,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,3,prb55384_14,1,-1
5,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,4,prb55384_14,4,8
6,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,5,prb55384_14,-1,2
7,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,6,prb55384_14,-1,7
8,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,7,prb55384_14,-1,6
9,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,8,prb55384_14,-1,-1


In [96]:
processed_data = parse_data_matrix(all_data_matrix)

In [97]:
processed_data_head = processed_data.head(200)
processed_data_head

,subject,event,move,instance,dist,p_as_str
0,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb55384_14,0,0
1,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb55384_14,1,3
2,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,1,prb55384_14,-2,3
3,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,2,prb55384_14,2,3
4,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,3,prb55384_14,1,-1
...,...,...,...,...,...,...
30,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,11,prb23404_14,3,6
31,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,12,prb23404_14,0,0
0,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb47495_14,0,0
1,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb47495_14,-1,2


In [99]:
processed_data.to_csv('../Data/my_processed_data/processed_data.csv')